# Option Pricing with RustQuant

RustQuant provides comprehensive option pricing tools including:

- **Black-Scholes-Merton** analytic pricing and Greeks
- **Monte Carlo** simulation-based pricing
- **Implied volatility** via Peter Jaeckel's "Let's Be Rational" method
- **Exotic options**: Asian, Barrier, Binary, Lookback, Power, and more

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }

## 1. Black-Scholes-Merton Pricing

The `BlackScholesMertonBuilder` provides analytic pricing and Greeks for European options.

In [ ]:
use time::macros::date;
use RustQuant::instruments::options::*;

let call = BlackScholesMertonBuilder::default()
    .underlying_price(100.0)
    .strike_price(100.0)
    .volatility(0.20)
    .risk_free_rate(0.05)
    .cost_of_carry(0.05)
    .expiration_date(date!(2027 - 03 - 21))
    .option_type(TypeFlag::Call)
    .build()
    .unwrap();

println!("=== European Call Option (ATM) ===");
println!("Price  = {:.4}", call.price());
println!("Delta  = {:.4}", call.delta());
println!("Gamma  = {:.4}", call.gamma());
println!("Theta  = {:.4}", call.theta());
println!("Vega   = {:.4}", call.vega());
println!("Rho    = {:.4}", call.rho());

## 2. Put Option and Put-Call Parity

In [ ]:
let put = BlackScholesMertonBuilder::default()
    .underlying_price(100.0)
    .strike_price(100.0)
    .volatility(0.20)
    .risk_free_rate(0.05)
    .cost_of_carry(0.05)
    .expiration_date(date!(2027 - 03 - 21))
    .option_type(TypeFlag::Put)
    .build()
    .unwrap();

println!("Call Price = {:.4}", call.price());
println!("Put Price  = {:.4}", put.price());
println!();

// Put-Call Parity: C - P = S - K * exp(-rT)
let parity_lhs = call.price() - put.price();
let t = 1.0; // approximate
let parity_rhs = 100.0 - 100.0 * (-0.05_f64 * t).exp();
println!("Put-Call Parity Check:");
println!("  C - P           = {:.4}", parity_lhs);
println!("  S - K*exp(-rT)  = {:.4} (approx)", parity_rhs);

## 3. Implied Volatility

Given an observed market price, compute the implied volatility using
Peter Jaeckel's highly accurate "Let's Be Rational" algorithm.

In [ ]:
// Start with an option priced at 20% vol
let market_price = call.price();
println!("Market price (from 20% vol): {:.4}", market_price);

// Recover the implied volatility
let iv = call.implied_volatility(market_price);
println!("Implied volatility: {:.6} (expected ~0.20)", iv);

## 4. Volatility Smile

Compute option prices across different strikes to see how Greeks change.

In [ ]:
println!("{:<10} {:<12} {:<10} {:<10}", "Strike", "Price", "Delta", "Gamma");
println!("{}", "-".repeat(42));

for strike in (80..=120).step_by(5) {
    let opt = BlackScholesMertonBuilder::default()
        .underlying_price(100.0)
        .strike_price(strike as f64)
        .volatility(0.20)
        .risk_free_rate(0.05)
        .cost_of_carry(0.05)
        .expiration_date(date!(2027 - 03 - 21))
        .option_type(TypeFlag::Call)
        .build()
        .unwrap();

    println!("{:<10} {:<12.4} {:<10.4} {:<10.6}", strike, opt.price(), opt.delta(), opt.gamma());
}

## 5. Monte Carlo Option Pricing

Price options using Monte Carlo simulation. This works for both vanilla and exotic options.

In [ ]:
use RustQuant::instruments::*;
use RustQuant::stochastics::*;

let underlying = 100.0;
let strike = 100.0;
let rate = 0.05;
let volatility = 0.20;
let expiry = date!(2027 - 03 - 21);

// Set up the stochastic process (GBM)
let process = GeometricBrownianMotion::new(rate, volatility);
let config = StochasticProcessConfig::new(
    underlying, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama, 50_000, false, None
);

// Vanilla European Call
let vanilla = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Call);
let mc_price = vanilla.price_monte_carlo(&process, &config, rate);
println!("Monte Carlo Call Price: {:.4} (analytic: {:.4})", mc_price, call.price());

## 6. Exotic Options: Asian Option

Asian options have payoffs depending on the average price over the option's life.

In [ ]:
let contract = OptionContractBuilder::default()
    .type_flag(TypeFlag::Call)
    .exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Fixed))
    .build()
    .unwrap();

let asian = AsianOption::new(
    contract.clone(),
    AveragingMethod::ArithmeticDiscrete,
    Some(strike),
);

let asian_price = asian.price_monte_carlo(&process, &config, rate);
println!("Asian Call Price (arithmetic avg): {:.4}", asian_price);
println!("Vanilla Call Price:                {:.4}", mc_price);
println!("\nNote: Asian options are cheaper due to averaging reducing volatility.");

## 7. Power Options

Power options have payoffs raised to a power, amplifying the leverage.

In [ ]:
let power = PowerOption::new(contract.clone(), strike, 2.0);
let power_price = power.price_monte_carlo(&process, &config, rate);
println!("Power Option Price (power=2): {:.4}", power_price);

## Summary

| Method | Use Case |
|--------|----------|
| Black-Scholes-Merton | Analytic European option pricing + Greeks |
| Monte Carlo | Path-dependent and exotic options |
| Implied Volatility | Calibration from market prices |

RustQuant also supports Barrier, Binary, Lookback, Forward Start, Gap, Log,
and Supershare options, as well as bond pricing (zero-coupon, coupon, Vasicek, Hull-White, CIR).